# LoRA Scan Sweep on Google Colab

⚠️ **Important:** This notebook requires persistent storage! Use `run_sweep_colab_persistent.ipynb` instead for:
- Auto-save to Google Drive after each run
- Resume after disconnections
- Early stopping to reduce training time

This notebook is for quick testing only.

**Setup:**
1. Runtime → Change runtime type
2. Select GPU (T4, L4, or A100)
3. Click Save

In [2]:
# Check GPU availability
!nvidia-smi

Wed Oct  7 00:02:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Clone the repository
!git clone https://github.com/ubitquitin/lora_scan.git
%cd lora_scan

Cloning into 'lora_scan'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 74 (delta 16), reused 73 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 31.94 KiB | 15.97 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/lora_scan


In [4]:
# Install micromamba (lightweight conda alternative)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba shell init -s bash -p ~/micromamba

bin/micromamba
init: The following argument was not expected: -p
Run with --help for more information.


In [5]:
# Create environment and install dependencies
!./bin/micromamba create -n spar python=3.12 -y -c conda-forge
!./bin/micromamba run -n spar pip install -e .

[+] 0.0s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                     ✔ Done (7.1 sec)

Resolving Environment                                                                     ✔ Done (0.2 sec)

Transaction

  Prefix: /root/.local/share/mamba/envs/spar

  Updating specs:

   - python=3.12
   - pip


  Package               Version  Build                 Channel          Size
──────────────────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge      29kB
  + bzip2                 1.0.8  hda65f42_10           conda-forge     258kB
  + ca-certificates   2026.7.22  hbd8a1cb_0           

In [13]:
# Run a test to verify setup
!./bin/micromamba run -n spar python scripts/run.py configs/experiments/sft_sweep_mmlu_gsm8k.yaml --dry-run | head -20

[1/72] TODO 4ec498d708  sft_sweep_multi_qwen25-0.5b_gsm8k_chat  seed=42  location=late layers=[16, 17, 18, 19, 20, 21, 22, 23]
[2/72] TODO ff95d01a8b  sft_sweep_multi_qwen25-0.5b_gsm8k_chat  seed=123  location=late layers=[16, 17, 18, 19, 20, 21, 22, 23]
[3/72] TODO d2a32dcf96  sft_sweep_multi_qwen25-0.5b_gsm8k_chat  seed=456  location=late layers=[16, 17, 18, 19, 20, 21, 22, 23]
[4/72] TODO e35e44b3f3  sft_sweep_multi_qwen25-0.5b_gsm8k_cot  seed=42  location=late layers=[16, 17, 18, 19, 20, 21, 22, 23]
[5/72] TODO a8a301ca94  sft_sweep_multi_qwen25-0.5b_gsm8k_cot  seed=123  location=late layers=[16, 17, 18, 19, 20, 21, 22, 23]
[6/72] TODO be9a4cdc53  sft_sweep_multi_qwen25-0.5b_gsm8k_cot  seed=456  location=late layers=[16, 17, 18, 19, 20, 21, 22, 23]
[7/72] TODO e6ca472c60  sft_sweep_multi_qwen25-0.5b_gsm8k_answer_only  seed=42  location=late layers=[16, 17, 18, 19, 20, 21, 22, 23]
[8/72] TODO 024b5bd617  sft_sweep_multi_qwen25-0.5b_gsm8k_answer_only  seed=123  location=late layers=[

## Authenticate with Hugging Face

Required for gated models like Gemma 2:

In [ ]:
# Login to Hugging Face (for gated models like Gemma 2)
from huggingface_hub import login
from getpass import getpass

# Get your token from: https://huggingface.co/settings/tokens
hf_token = getpass("Enter your HF token: ")
if hf_token:
    login(token=hf_token)
    print("✓ Logged in to Hugging Face")
else:
    print("⚠️  No token provided. Gated models (Gemma 2) will fail.")

In [ ]:
# Run a single experiment for testing (recommended)
# Test with just one run before committing to the full sweep
!./bin/micromamba run -n spar python scripts/run.py configs/experiments/sft_sweep_mmlu_gsm8k.yaml --only "qwen25-0.5b_gsm8k_chat" --seeds 42

## ⚠️ Full Sweep (Use Persistent Notebook Instead!)

**Warning:** The full sweep takes ~33 hours with early stopping. If Colab disconnects, you'll lose all progress!

**Recommended:** Use `run_sweep_colab_persistent.ipynb` which auto-saves to Google Drive after each run.

In [ ]:
# Download test results
!tar -czf test_results.tar.gz runs/
from google.colab import files
files.download('test_results.tar.gz')

## Analyze Results

In [ ]:
# Load and display results
import sys
sys.path.insert(0, 'src')

from lorascan import runs
import pandas as pd

results = runs.collect_results()
df = pd.DataFrame(results)
print(f"\nCompleted runs: {len(df)}/72")
if len(df) > 0:
    print(df.head())
    
    # Summary statistics
    print("\n=== Results Summary ===")
    print(df.groupby(['model', 'dataset', 'prompt'])['accuracy'].agg(['mean', 'std', 'count']))
else:
    print("No completed runs yet")